In [ ]:
# ──── 1: Install ──────────────────────────────────────────
!pip install -q faiss-cpu bitsandbytes sentence-transformers datasets transformers accelerate
from huggingface_hub import login
login()

# ──── 2: Build corpus & FAISS index ───────────────────────
import random, pandas as pd, faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

paradetox = load_dataset("s-nlp/paradetox", split="train")
non_toxic_corpus = list(set(paradetox["en_neutral_comment"]))
print(f"Full unique corpus: {len(non_toxic_corpus)}")

random.seed(42)
non_toxic_corpus = random.sample(non_toxic_corpus, 5000)
print(f"Sampled corpus: {len(non_toxic_corpus)}")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
corpus_embeddings = embedder.encode(
    non_toxic_corpus, batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)
print("Index built.")

# ──── 3: Load model ───────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "UBC-NLP/DetoxLLM-7B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
print("Model loaded.")

# ──── 4: Pipeline functions ───────────────────────────────
def generate_rationale(toxic_comment, max_new_tokens=150):
    prompt = (
        "Rewrite the following toxic input into non-toxic version. Let's break the input "
        "down step by step to rewrite the non-toxic version. You should first think about "
        "the explanation of why the input text is toxic. Then generate the detoxic output. "
        "You must preserve the original meaning as much as possible.\nInput: "
        + toxic_comment + "\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    full_text = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):]
    return full_text.split("Output:")[0].strip()

def retrieve_examples(toxic_comment, rationale, k_retrieve=10, k_return=3, threshold=0.25):
    # Step 1: Broad retrieval using combined query
    query = f"{toxic_comment} {rationale}"
    q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    _, idx = index.search(q, k_retrieve)
    candidates = [non_toxic_corpus[i] for i in idx[0]]

    # Step 2: Rerank candidates by similarity to original toxic comment
    import numpy as np
    original_emb = embedder.encode([toxic_comment], normalize_embeddings=True, convert_to_numpy=True)
    candidate_embs = embedder.encode(candidates, normalize_embeddings=True, convert_to_numpy=True)
    scores = candidate_embs @ original_emb[0]
    ranked = sorted(zip(scores, candidates), reverse=True)

    # Step 3: Return top k_return above threshold
    return [ex for score, ex in ranked[:k_return] if score >= threshold]

def generate_rewrite(toxic_comment, rationale, examples, max_new_tokens=80):
    if examples:
        examples_block = "\n".join(f"{i+1}. {e}" for i, e in enumerate(examples))
        examples_section = f"Non-toxic style examples:\n{examples_block}\n\n"
    else:
        examples_section = ""   # skip the examples block entirely
    prompt = (
        "Below is a toxic comment and an explanation of why it is toxic. "
        "Rewrite the comment to remove toxicity while preserving every "
        "important idea and intent from the original comment.\n\n"
        f'Toxic Comment: "{toxic_comment}"\n'
        f"Explanation: {rationale}\n\n"
        f"{examples_section}Non-toxic Rewrite:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):]
    return text.strip().split("\n")[0].strip()

print("System ready.")

# ──── 5: Run ──────────────────────────────────────────────
while True:
    toxic_comment = input("\nEnter toxic comment (or 'exit'): ")
    if toxic_comment.lower() == "exit":
        print("Goodbye!")
        break

    rationale = generate_rationale(toxic_comment)
    print(f"\nRationale:\n{rationale}")

    retrieved = retrieve_examples(toxic_comment, rationale,)
    print("\nRetrieved Examples:")
    for i, ex in enumerate(retrieved, 1):
        print(f"  {i}. {ex}")

    rewrite = generate_rewrite(toxic_comment, rationale, retrieved)
    print(f"\nDetoxified Rewrite:\n{rewrite}")
    print("-" * 60)

In [ ]:
# ──── 6: Multi-system Evaluation ──────────────────────────
!pip install -q evaluate bert-score detoxify

import evaluate
from bert_score import score as bert_score
from detoxify import Detoxify
from datasets import load_dataset
import pandas as pd
import numpy as np

# ── Load evaluation tools ─────────────────────────────────────
bleu_metric = evaluate.load("bleu")
detox_model = Detoxify('original')

# ── Load 100 samples ──────────────────────────────────────────
raw_df = load_dataset("s-nlp/paradetox", split="train").to_pandas()
raw_df = raw_df[raw_df["en_toxic_comment"].str.split().str.len() >= 5].reset_index(drop=True)

np.random.seed(42)
sample_df = raw_df.sample(100).reset_index(drop=True)
print(f"Evaluation samples: {len(sample_df)}")

# ── Helper: compute metrics ───────────────────────────────────
def compute_metrics(rewrites, references, toxic_inputs):
    bleu_scores, sim_scores, sta_scores, tox_deltas = [], [], [], []

    for rewrite, ref, toxic in zip(rewrites, references, toxic_inputs):
        bleu_scores.append(
            bleu_metric.compute(predictions=[rewrite], references=[[ref]])["bleu"]
        )
        embs = embedder.encode([toxic, rewrite], normalize_embeddings=True)
        sim_scores.append(float(np.dot(embs[0], embs[1])))

        orig_tox = detox_model.predict(toxic)["toxicity"]
        rew_tox  = detox_model.predict(rewrite)["toxicity"]
        sta_scores.append(int(rew_tox < 0.5))
        tox_deltas.append(orig_tox - rew_tox)

    _, _, F1 = bert_score(rewrites, references, lang="en", verbose=False)

    return {
        "BLEU":           round(np.mean(bleu_scores), 4),
        "BERT F1":        round(np.mean([f.item() for f in F1]), 4),
        "SIM":            round(np.mean(sim_scores), 4),
        "STA (%)":        round(np.mean(sta_scores) * 100, 2),
        "Toxicity Delta": round(np.mean(tox_deltas), 4),
    }

# ── System 1: DetoxLLM only (no RAG) ─────────────────────────
print("\nRunning DetoxLLM (no RAG)...")

def detoxllm_only(toxic_comment):
    prompt = (
        "Rewrite the following toxic input into non-toxic version. Let's break the input "
        "down step by step to rewrite the non-toxic version. You should first think about "
        "the explanation of why the input text is toxic. Then generate the detoxic output. "
        "You must preserve the original meaning as much as possible.\nInput: "
        + toxic_comment + "\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200,
                         do_sample=False, pad_token_id=tokenizer.eos_token_id)
    full_text = tokenizer.decode(out[0], skip_special_tokens=True)[len(prompt):]
    if "Detoxification:" in full_text:
        return full_text.split("Detoxification:")[-1].strip().split("\n")[0].strip()
    return full_text.strip().split("\n")[0].strip()

rewrites_no_rag = []
for i, row in sample_df.iterrows():
    rewrites_no_rag.append(detoxllm_only(row["en_toxic_comment"]))
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 done...")

# ── System 2: DetoxLLM + RAG ─────────────────
print("\nRunning DetoxLLM + RAG...")

rewrites_rag = []
for i, row in sample_df.iterrows():
    toxic     = row["en_toxic_comment"]
    rationale = generate_rationale(toxic)
    retrieved = retrieve_examples(toxic, rationale)
    rewrites_rag.append(generate_rewrite(toxic, rationale, retrieved))
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 done...")

# ── Compute metrics ───────────────────────────────────────────
toxics     = sample_df["en_toxic_comment"].tolist()
references = sample_df["en_neutral_comment"].tolist()

print("\nComputing metrics...")
metrics_no_rag = compute_metrics(rewrites_no_rag, references, toxics)
metrics_rag    = compute_metrics(rewrites_rag,    references, toxics)
metrics_human  = compute_metrics(references,      references, toxics)

# ── Summary table ─────────────────────────────────────────────
summary = pd.DataFrame({
    "Metric":                list(metrics_rag.keys()),
    "DetoxLLM (no RAG)":     list(metrics_no_rag.values()),
    "DetoxLLM + RAG": list(metrics_rag.values()),
    "Human Reference":       list(metrics_human.values()),
})

print("\n── Summary Table ───────────────────────────────────────")
print(summary.to_string(index=False))